# Research Agent RL — Week 2: Baseline Agent Evaluation

**Setup:** GPU T4 x1, Internet ON (same as Week 1)

This notebook:
1. Clones the repo and installs dependencies
2. Downloads the Week 1 SFT checkpoint via Kaggle API
3. Builds the tool index from HotpotQA
4. Runs 3 stopping policies and compares them

## 1. Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — enable T4 in Settings!')

## 2. Clone repo & install

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## 3. Restore Week 1 checkpoint

The checkpoint was saved as a kernel output in Week 1 and is available via `kaggle kernels output`.

In [ ]:
import os, shutil

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

if not os.path.exists(ADAPTER_DIR):
    print('Downloading Week 1 checkpoint from kernel output...')
    !mkdir -p /tmp/w1_out
    !kaggle kernels output wuyue22/week1-sft -p /tmp/w1_out
    # Try the top-level adapter copy first, then nested path
    candidates = [
        '/tmp/w1_out/qwen-sft-adapter',
        '/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/final',
    ]
    src = None
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'adapter_config.json')):
            src = c
            break
    if src:
        os.makedirs('checkpoints/qwen-sft', exist_ok=True)
        shutil.copytree(src, ADAPTER_DIR)
        print(f'Adapter copied from {src} → {ADAPTER_DIR}')
    else:
        import glob
        ckpts = sorted(glob.glob('/tmp/w1_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        if ckpts:
            shutil.copytree(ckpts[-1], ADAPTER_DIR)
            print(f'Using latest checkpoint: {ckpts[-1]}')
        else:
            print('[ERROR] No checkpoint found in kernel output.')
else:
    print(f'Checkpoint already present at {ADAPTER_DIR}')

!ls -lh {ADAPTER_DIR}

## 4. Load model

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ADAPTER_DIR = 'checkpoints/qwen-sft/final'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

print(f'Loading base model: {base_model_name}')
base = AutoModelForCausalLM.from_pretrained(
    base_model_name, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
model.config.use_cache = True

allocated = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded. VRAM: {allocated:.2f} GB')

## 5. Load HotpotQA & build tool index

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_QUESTIONS = 100   # 100 is enough for meaningful baselines; keeps runtime under 2h

print('Loading HotpotQA validation split...')
hf = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
val = hf['validation']

questions = [
    {'question': val[i]['question'], 'answer': val[i]['answer'], 'context': val[i]['context']}
    for i in range(min(N_QUESTIONS, len(val)))
]
print(f'Loaded {len(questions)} questions')

# Build a document-level tool index from HotpotQA titles + sentences.
executor = ToolExecutor(top_k=2)
executor.build_from_hotpotqa(val, index_path='data/tool_index.jsonl')
print(f'Tool index size: {len(executor)} documents')

## 6. Smoke-test the agent (1 question)

In [ ]:
from agent.agent import ResearchAgent
from agent.stopping import ConfidencePolicy

agent = ResearchAgent(model, tokenizer, executor, ConfidencePolicy(threshold=0.8), max_steps=6)
q = questions[0]
result = agent.run(q['question'], gold_answer=q['answer'])

print(f'Q : {result.question}')
print(f'Gold   : {result.gold_answer}')
print(f'Pred   : {result.pred_answer}')
print(f'Correct: {result.correct}  |  Steps: {result.n_steps}  |  Tools: {result.n_tool_calls}  |  Stopped by: {result.stopped_by}')
print()
for rec in result.steps:
    print(f'  Step {rec.step_idx+1} [{rec.action}] conf={rec.confidence}  thought="{rec.thought[:60]}..."')
    if rec.tool_output:
        print(f'    → tool output: {rec.tool_output[:120]}...')

## 7. Evaluate all baseline policies

In [ ]:
from tqdm import tqdm
from agent.stopping import FixedStepPolicy, ConfidencePolicy, NeverStop
from eval.metrics import compute_metrics, compare_policies

policies = {
    'FixedStep (N=2)':          FixedStepPolicy(max_steps=2),
    'FixedStep (N=3)':          FixedStepPolicy(max_steps=3),
    'Confidence (τ=0.75)':      ConfidencePolicy(threshold=0.75),
    'Confidence (τ=0.85)':      ConfidencePolicy(threshold=0.85),
    'NeverStop (max_steps=6)':  NeverStop(),
}

all_results = {}

for name, policy in policies.items():
    print(f'\n--- {name} ---')
    agent = ResearchAgent(model, tokenizer, executor, policy, max_steps=6)
    results = []
    for q in tqdm(questions, desc=name):
        policy.reset()
        results.append(agent.run(q['question'], gold_answer=q['answer']))
    all_results[name] = results
    m = compute_metrics(results)
    print(f"  acc={m['accuracy']:.3f}  steps={m['avg_steps']:.1f}  tools={m['avg_tool_calls']:.1f}  eff={m['efficiency_score']:.4f}")

## 8. Results table

In [ ]:
print(compare_policies(all_results))

## 9. Save results

In [ ]:
import json, os

os.makedirs('eval', exist_ok=True)
serializable = {}
for name, results in all_results.items():
    serializable[name] = [
        {'question': r.question, 'gold_answer': r.gold_answer,
         'pred_answer': r.pred_answer, 'correct': r.correct,
         'n_steps': r.n_steps, 'n_tool_calls': r.n_tool_calls,
         'stopped_by': r.stopped_by}
        for r in results
    ]

results_path = '/kaggle/working/baseline_results.json'
with open(results_path, 'w') as f:
    json.dump(serializable, f, indent=2)
print(f'Results saved → {results_path}')
print('Download from the Output tab or via: kaggle kernels output wuyue22/rl-sft-research')